# 04b — Social Media Charts

Publication-ready charts for @unwelcomedata. Pillow pipeline (no Altair/vl-convert).

**Final curated set:**
1. Felony threshold choropleth + bar legend
2. Felony lookback period choropleth + bar legend
3. First-time vs repeat offenders — stacked 100% bar
4. DUI enforcement: checkpoints × vehicle impound — bivariate matrix

In [ ]:
import sys, os
from pathlib import Path
import pandas as pd
import numpy as np
import duckdb
import geopandas as gpd
import yaml

PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
sys.path.insert(0, str(PROJECT.parent / 'shared'))

from chart_factory import render_chart
from viz import PALETTE, PRESETS

with open(PROJECT / 'config.yaml') as f:
    cfg = yaml.safe_load(f)

df = pd.read_parquet(PROJECT / 'export' / 'dui_by_state_v4.parquet')
con = duckdb.connect(str(PROJECT / 'data' / 'project.duckdb'), read_only=True)

# Derived columns
def felony_label(row):
    if row['has_felony_dui'] == 0: return 'Always a misdemeanor'
    t = row['felony_dui_threshold']
    if pd.isna(t): return 'Always a misdemeanor'
    t = int(t)
    if t == 2: return 'Felony on 2nd'
    elif t == 3: return 'Felony on 3rd'
    elif t == 4: return 'Felony on 4th'
    return f'Felony on {t}th'

df['felony_at'] = df.apply(felony_label, axis=1)

# Load shapefile (used by charts 1, 2, 4)
shapefile = PROJECT / 'data' / 'raw' / 'geo' / 'cb_2022_us_state_20m.shp'
geo_base = gpd.read_file(shapefile)
geo_base = geo_base[~geo_base['STATEFP'].isin({'60','66','69','72','78'})].copy()

print(f'Project: {PROJECT.name}, {df.shape[0]} states x {df.shape[1]} cols')

---
## 1. Felony Threshold Map

In [ ]:
# Merge felony_at onto geodataframe
map_data1 = df[['state_fips', 'felony_at']].copy()
map_data1['state_fips'] = map_data1['state_fips'].astype(str).str.zfill(2)
geo1 = geo_base.merge(map_data1, left_on='STATEFP', right_on='state_fips', how='left')

felony_categories = ['Always a misdemeanor', 'Felony on 2nd', 'Felony on 3rd', 'Felony on 4th']
felony_colors = {
    'Always a misdemeanor': '#D9D9D9',
    'Felony on 2nd': '#E76F51',
    'Felony on 3rd': '#2A9D8F',
    'Felony on 4th': '#264653',
}

render_chart({
    'type': 'choropleth',
    'preset': 'twitter_landscape',
    'geo': geo1,
    'category_col': 'felony_at',
    'categories': felony_categories,
    'colors': felony_colors,
    'title': 'When Does a DUI Become a Felony?',
    'subtitle': 'Number of DUI convictions that triggers a felony charge, by state',
    'source': 'NASID state enforcement laws, corroborated with NCSL | CA is a wobbler (felony at prosecutor discretion on 4th+)',
    'bar_legend_title': 'States per category',
    'filename': 'social_map_felony_threshold',
    'cfg': cfg,
})

---
## 2. Felony Lookback Period Map

In [ ]:
# Bin lookback into categories
def lookback_bin(row):
    if row['has_felony_dui'] == 0: return 'No felony law'
    yrs = row['lookback_years']
    if yrs <= 7: return 'Short (5\u20137 yr)'
    elif yrs <= 15: return 'Standard (10\u201315 yr)'
    else: return 'Lifetime'

df['lookback_cat'] = df.apply(lookback_bin, axis=1)

map_data2 = df[['state_fips', 'lookback_cat']].copy()
map_data2['state_fips'] = map_data2['state_fips'].astype(str).str.zfill(2)
geo2 = geo_base.merge(map_data2, left_on='STATEFP', right_on='state_fips', how='left')

lookback_categories = ['No felony law', 'Short (5\u20137 yr)', 'Standard (10\u201315 yr)', 'Lifetime']
lookback_colors = {
    'No felony law': '#D9D9D9',
    'Short (5\u20137 yr)': '#E76F51',
    'Standard (10\u201315 yr)': '#2A9D8F',
    'Lifetime': '#264653',
}

render_chart({
    'type': 'choropleth',
    'preset': 'twitter_landscape',
    'geo': geo2,
    'category_col': 'lookback_cat',
    'categories': lookback_categories,
    'colors': lookback_colors,
    'title': 'How Long Is the Window to Trigger a DUI Felony?',
    'subtitle': 'Prior convictions only count toward felony escalation within this lookback period',
    'source': 'State DUI statutes (ailawyer.pro, NASID, WA SB 5032) | 99 = lifetime; DC, MD, NJ have no felony DUI law',
    'bar_legend_title': 'States per category',
    'filename': 'social_map_felony_lookback',
    'cfg': cfg,
})

---
## 3. First-Time vs Repeat Offenders

In [ ]:
# Aggregate prior-DWI data by felony threshold group
prior = con.sql('SELECT state_fips, impaired_drivers_known_history, impaired_with_prior_dwi FROM fars_prior_dwi_speed').df()
prior['state_fips'] = prior['state_fips'].astype(str).str.zfill(2)
merged = prior.merge(df[['state_fips', 'felony_at']], on='state_fips', how='left')

order = ['Always a misdemeanor', 'Felony on 4th', 'Felony on 3rd', 'Felony on 2nd']
group_agg = merged.groupby('felony_at').agg(
    total_impaired=('impaired_drivers_known_history', 'sum'),
    total_with_prior=('impaired_with_prior_dwi', 'sum')
).reindex(order)
group_agg['pct_repeat'] = (group_agg['total_with_prior'] / group_agg['total_impaired'] * 100).round(1)
group_agg['pct_first_time'] = (100 - group_agg['pct_repeat']).round(1)
group_agg['n_states'] = merged.groupby('felony_at')['state_fips'].nunique().reindex(order)

# National total
nat_repeat = group_agg['total_with_prior'].sum() / group_agg['total_impaired'].sum() * 100

# Build chart DataFrame
rows = []
for cat in order:
    row = group_agg.loc[cat]
    label = f"{cat} ({int(row['n_states'])} states)"
    rows.append({'group': label, 'pct_first_time': row['pct_first_time'], 'pct_repeat': row['pct_repeat']})

rows.append({'group': 'National (51 states)', 'pct_first_time': round(100 - nat_repeat, 1), 'pct_repeat': round(nat_repeat, 1)})
bar_df = pd.DataFrame(rows)

render_chart({
    'type': 'stacked_100pct',
    'preset': 'twitter_landscape',
    'table': bar_df,
    'group_col': 'group',
    'segments': [
        {'col': 'pct_first_time', 'label': 'No prior DUI conviction', 'color': '#2A9D8F'},
        {'col': 'pct_repeat', 'label': 'Had prior DUI conviction', 'color': '#E76F51'},
    ],
    'title': 'Most DUI Deaths Involve Drivers With No Prior DUI Conviction',
    'subtitle': 'States grouped by felony-charge DUI threshold. No prior conviction \u2260 first time driving impaired.',
    'source': 'NHTSA FARS 2024 (PREV_DWI field) | MS reports 0% prior DWI (likely undercount)',
    'bar_height': 50,
    'bar_gap': 24,
    'filename': 'social_first_time_vs_repeat_x',
    'cfg': cfg,
})

---
## 4. DUI Enforcement: Checkpoints \u00d7 Vehicle Impound

In [ ]:
# Load impound data and build enforcement quadrants
df_impound = pd.read_parquet(PROJECT / 'data' / 'interim' / 'vehicle_impound_laws.parquet')
df_impound['has_mandatory_impound'] = (
    (df_impound['vehicle_impound_law'] == 'yes') &
    (df_impound['impound_mandatory'] == 'mandatory')
).astype(int)

df4 = df[['state_fips', 'state_abbr', 'state_name', 'checkpoints_permitted']].copy()
df4 = df4.merge(df_impound[['state_fips', 'has_mandatory_impound']], on='state_fips', how='left')
df4['has_mandatory_impound'] = df4['has_mandatory_impound'].fillna(0).astype(int)

def impound_quad(row):
    cp = row['checkpoints_permitted'] == 1
    imp = row['has_mandatory_impound'] == 1
    if cp and imp: return 'Both: checkpoints + impound'
    elif cp and not imp: return 'Checkpoints only'
    elif not cp and imp: return 'Impound only (no checkpoints)'
    else: return 'Neither'

df4['enforcement_quad'] = df4.apply(impound_quad, axis=1)

QUAD_COLORS = {
    'Both: checkpoints + impound': '#264653',
    'Impound only (no checkpoints)': '#2A9D8F',
    'Checkpoints only': '#E76F51',
    'Neither': '#D9D9D9',
}

# Merge onto geodataframe
map_data4 = df4[['state_fips', 'enforcement_quad']].copy()
map_data4['state_fips'] = map_data4['state_fips'].astype(str).str.zfill(2)
geo4 = geo_base.merge(map_data4, left_on='STATEFP', right_on='state_fips', how='left')

render_chart({
    'type': 'choropleth_bivariate',
    'preset': 'twitter_landscape',
    'geo': geo4,
    'category_col': 'enforcement_quad',
    'matrix_categories': [
        ['Neither', 'Checkpoints only'],
        ['Impound only (no checkpoints)', 'Both: checkpoints + impound'],
    ],
    'colors': QUAD_COLORS,
    'axis_labels': ('Checkpoints \u2192', 'Vehicle impound \u2192'),
    'title': 'Can They Stop You AND Take Your Car?',
    'subtitle': 'Checkpoints = sobriety checkpoints legal. Impound = mandatory vehicle seizure upon DUI.',
    'source': 'NASID, NHTSA, state statutes | Dark: both tools. Gray: neither (TX, ID, MT, AK, WY).',
    'filename': 'social_bivariate_enforcement',
    'cfg': cfg,
})

---
## Done

Four charts exported to `outputs/social/`.

In [ ]:
con.close()
print('\u2713 All charts rendered. Pipeline: Pillow (no Altair/vl-convert).')